# Corpus stats

In [1]:
from shared import label2id

In [2]:
import pandas as pd

df = pd.read_json('data/main_corpus.jsonl', lines=True)
df.head()

,Unnamed: 0,uuid,clean_comment_text,source,ts,label,__proba,tokens,spans,words,labels,probas,unique_tags,if_csw,labels_str
0,7,jtYRtKekkbPELEizm2xHW5,"Следаки,опера кормушка гой",telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.909292,"[Следаки, ,, опера, кормушка, гой]","[[0, 7], [7, 8], [8, 13], [14, 22], [23, 26]]","[Следаки, ,, опера, кормушка, гой]","[5, 7, 0, 5, 6]","[0.9070628285000001, 0.9981331229, 0.964494168...","[0, 5, 6, 7]",True,"[ru, univ, ambig, ru, skz]"
1,17,RNjaGp4N8cytGDopdtLVHt,"Уже привыкли дп Алматы лапша гой , клевета пос...",telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.969197,"[Уже, привыкли, дп, Алматы, лапша, гой, ,, кле...","[[0, 3], [4, 12], [13, 15], [16, 22], [23, 28]...","[Уже, привыкли, дп, Алматы, лапша, гой, ,, кле...","[5, 5, 0, 0, 5, 6, 7, 0, 5, 0, 0, 7, 0, 6, 5, ...","[0.9991756082000001, 0.9989144802000001, 0.818...","[0, 5, 6, 7]",True,"[ru, ru, ambig, ambig, ru, skz, univ, ambig, r..."
2,18,VRXPwZG9R4ownuEcuhjouY,Ювенальная Полиция по защите детей контроль жок,telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.997552,"[Ювенальная, Полиция, по, защите, детей, контр...","[[0, 10], [11, 18], [19, 21], [22, 28], [29, 3...","[Ювенальная, Полиция, по, защите, детей, контр...","[5, 0, 5, 5, 5, 0, 1]","[0.8887240887000001, 0.9452017546, 0.997068822...","[0, 1, 5]",True,"[ru, ambig, ru, ru, ru, ambig, kz]"
3,26,EDRyxaBZcjiEY5CgNuP5Ap,Мынау не ? Проверка? Барма?,telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.683077,"[Мынау, не, ?, Проверка, ?, Барма, ?]","[[0, 5], [6, 8], [9, 10], [11, 19], [19, 20], ...","[Мынау, не, ?, Проверка, ?, Барма, ?]","[1, 1, 7, 5, 7, 0, 7]","[0.9440215826, 0.531437993, 0.9991680384, 0.89...","[0, 1, 5, 7]",True,"[kz, kz, univ, ru, univ, ambig, univ]"
4,33,7kpEF4YzhpzjEHv2w47oQb,Құлағын кесіп тастады 667 в прям смысле этого ...,telegram__egovpress,2026-05-16 22:05:08.728025,__label__kaz_Cyrl,0.630114,"[Құлағын, кесіп, тастады, 667, в, прям, смысле...","[[0, 7], [8, 13], [14, 21], [22, 25], [26, 27]...","[Құлағын, кесіп, тастады, 667, в, прям, смысле...","[1, 1, 1, 7, 5, 5, 5, 5, 5]","[0.9971574545, 0.9972094893000001, 0.993018388...","[1, 5, 7]",True,"[kz, kz, kz, univ, ru, ru, ru, ru, ru]"


In [3]:
df.drop(df[df['words'].apply(len)!=df['tokens'].apply(len)].index, inplace=True)

In [4]:
# частота тегов
from collections import Counter
stats = Counter()

def get_stats(tags):
    """
    Посчитать количество для каждого тега
    """
    stats.update(tags)

df['labels_str'].apply(get_stats)
stats

Counter({'ru': 942937,
         'kz': 595790,
         'univ': 424019,
         'ambig': 73177,
         'skz': 31878,
         'other': 8534,
         'mixed_kz-ru': 3268,
         'mixed_ru-kz': 15})

In [5]:
# absolute tag frequency
pd.Series(stats)

ru             942937
univ           424019
ambig           73177
skz             31878
kz             595790
mixed_kz-ru      3268
other            8534
mixed_ru-kz        15
dtype: int64

In [6]:
# tag frequency in percents
pd.Series(stats) / stats.total() * 100

ru             45.341837
univ           20.389273
ambig           3.518771
skz             1.532878
kz             28.649012
mixed_kz-ru     0.157144
other           0.410364
mixed_ru-kz     0.000721
dtype: float64

In [7]:
print('Tokens total:', stats.total())

Tokens total: 2079618


In [8]:
# среднее значение вероятности (которую вернула модель) для каждого тега
proba_stats = {k: 0 for k in label2id.keys()}

def calc_mean_proba(row):
    """
    Считает сумму вероятностей для каждого тега (чтобы потом ее можно было разделить на абсолютную частоту тегов и получить среднее значение вероятности)
    """
    for l, p in zip(row['labels_str'], row['probas']):
        proba_stats[l] += p

df.apply(calc_mean_proba, axis=1)
print(pd.Series(proba_stats) / pd.Series(stats))

ambig          0.759017
kz             0.960449
mixed_kz-ru    0.412574
mixed_ru-kz    0.279016
other          0.737798
ru             0.976195
skz            0.598547
univ           0.990683
dtype: float64


In [9]:
# заменяем ambig, skz, mixed_*
def group_tags(lst):
    new_lst = []
    for i, tag_ in enumerate(lst):
        if i==0 and tag_ == 'ambig':
            new_lst.append(lst[i+1])
        elif tag_ == 'ambig':
            new_lst.append(lst[i-1])
        elif tag_=='skz':
            new_lst.append('kz')
        elif tag_=='mixed_ru-kz':
            new_lst.append('ru')
        elif tag_=='mixed_kz-ru':
            new_lst.append('kz')
        else:
            new_lst.append(tag_)
    assert len(lst)==len(new_lst)
    return new_lst

df['labels_str__grouped'] = df['labels_str'].apply(group_tags)

In [10]:
# Code-Mixing Index (CMI)
def calc_cmi(lst, ignored_tags=('univ')):
    """
    Считает Code-Mixing Index (CMI) -- доля языковых токенов, не принадлежащих матричному языку (ambig, univ не считаются)
    """
    cntr = Counter()
    cntr.update(list(filter(lambda x: x not in ignored_tags, lst)))

    _, matrix_lang_n = cntr.most_common(1)[0]

    if cntr:
        return 100 * (1 - matrix_lang_n/cntr.total())
    return 0

cmi = df['labels_str__grouped'].apply(calc_cmi)
cmi.describe()
# print('CMI =', cmi_counts['nested_tokens_count'] / cmi_counts['total'] * 100)


count    80002.000000
mean        20.161755
std         13.703793
min          0.000000
25%          8.695652
50%         16.666667
75%         30.769231
max         75.000000
Name: labels_str__grouped, dtype: float64

In [11]:
# Average switch-points (SP Avg)
def calc_sp(row, ignored_tags=('univ')):
    """
    Считает Average switch-points (SP Avg) -- среднее число интрасентенциональных (intra-sentential) переключений в корпусе.
    За границу предложения считаем точку, восклицательный и вопросительный знак, точку с запятой, перенос строки и эмодзи.
    Игнорируются теги 'univ'
    """
    res = []
    intrasent_csw = 0
    last_tag = 'univ'
    word_cntr = 0
    for i, (token_, tag_) in enumerate(zip(row['words'], row['labels_str__grouped'])):
        # print(i, token_, tag_)
        # конец предложения или конец документа
        if (token_ in ('.', '?', '!', ';', '\\n', '[EMOJI]') and word_cntr > 0) or i+1 == len(row['labels_str__grouped']):
            # print(intrasent_csw, 'sentence change', '='*20)
            res.append(intrasent_csw)
            intrasent_csw = 0
            word_cntr = 0
            last_tag = 'univ'
        # смена языка, если ни текущий, ни предыдущий токен не 'univ' или 'ambig'
        elif last_tag not in ignored_tags and tag_ not in ignored_tags and tag_ != last_tag:
            # print('>>> code switch')
            intrasent_csw += 1
            last_tag = tag_
            word_cntr += 1
        elif tag_ not in ignored_tags:
            last_tag = tag_
            word_cntr += 1
    return sum(res)

sp = df.apply(calc_sp, axis=1)
sp.describe()


count    80002.000000
mean         1.622734
std          1.621738
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         40.000000
dtype: float64

In [12]:
# Multilingual Index (M-index)
import numpy as np

stats = Counter()
df['labels_str__grouped'].apply(get_stats)
m_index_data = pd.Series(stats)

def calc_mindex(m_index_data, langs=['ru', 'kz', 'other']):
    """
    Считает отношение языковых тегов в корпусе (считаем только те, что перечислены в langs)
    """
    m_index_data = m_index_data.loc[langs]

    N = sum(m_index_data) # total number of words
    p = np.array(m_index_data) / N # proba

    m_index = (1 - np.sum(p**2)) / (np.sum(p**2) * (len(langs)-1))
    return m_index

print('M-index =', calc_mindex(m_index_data))

In [13]:
# Probability of Switching (I-index)
iindex_counts = {'language_tags_total': 0, 'csw_points': 0}
def calc_iindex(tags_col, lang_tags=['kz', 'ru', 'other']):
    """
    Считает отношение числа переключений в корпусе к количеству языковых тегов
    """
    last_tag = tags_col[0]
    for tag_ in tags_col:
        if tag_ in lang_tags:
            if last_tag != tag_:
                iindex_counts['csw_points'] += 1

            iindex_counts['language_tags_total'] += 1

df['labels_str__grouped'].apply(calc_iindex)
print('I-index =', iindex_counts['csw_points'] / iindex_counts['language_tags_total'])

M-index = 0.4722330324537372
I-index = 0.3247024104491991


In [14]:
# Burstiness
burstiness_counts = []
def calc_burstiness(tag_lst_, lang_tags=['kz', 'ru', 'other']):
    """
    Считает длину "островов" для каждого языка и складывает в словарь
    """
    tag_lst = list(filter(lambda x: x in lang_tags, tag_lst_))
    last_tag = tag_lst[0]
    word_cntr = 0
    for tag_ in tag_lst:
        if tag_ != last_tag:
            burstiness_counts.append(word_cntr)
            word_cntr = 1
        else:
            word_cntr += 1
        last_tag = tag_

    burstiness_counts.append(word_cntr)

df['labels_str__grouped'].apply(calc_burstiness)
burstiness_counts = pd.Series(burstiness_counts)

In [15]:
burstiness_counts.describe()

count    240141.000000
mean          6.764647
std          13.177889
min           1.000000
25%           1.000000
50%           2.000000
75%           6.000000
max         253.000000
dtype: float64

In [16]:
burstiness = (burstiness_counts.std() - burstiness_counts.mean()) / (burstiness_counts.std() + burstiness_counts.mean())
print('Burstiness =', burstiness)

Burstiness = 0.32158611983109464


In [17]:
# Language Entropy (LE)
stats = Counter()
df['labels_str__grouped'].apply(get_stats)
le_data = pd.Series(stats)

def get_term_(lang_tag, le_counts):
    p_ = le_counts[lang_tag]/le_counts.sum()
    return p_ * np.log2(p_)


def calc_le(le_data, lang_tags=['kz', 'ru', 'other']):
    """
    Считает энтропию языковых тегов
    """
    le_data_ = le_data.loc[lang_tags]
    max_bound = np.log2(len(lang_tags))
    return - sum([get_term_(l_, le_data_) for l_ in lang_tags]), max_bound

le, max_bound = calc_le(le_data, lang_tags=['kz', 'ru', 'other'])
print('Language Entropy =', le, f'(max={max_bound})')

Language Entropy = 1.0145457744583681 (max=1.584962500721156)


In [18]:
# Span Entropy (SE) (на основе burstiness)

def calc_se(span_lst):
    """
    Считает энтропию длинн сегментов на одном языке
    """
    res = 0
    span_lst = pd.Series(span_lst).value_counts()
    for l in range(1, max(span_lst)+1):
        if l not in span_lst.index:
            pass
        else:
            proba_ = span_lst[l] / span_lst.sum()
            res += proba_ * np.log2(proba_)
    return - res

burstiness_counts = []
df['labels_str__grouped'].apply(calc_burstiness, lang_tags=['kz', 'ru', 'other'])
burstiness_counts = pd.Series(burstiness_counts)

calc_se(burstiness_counts)

np.float64(3.6837511825107367)